<a href="https://colab.research.google.com/github/zaoa3345678-arch/Aaron-project_2/blob/main/1151_W07_PartB_%E5%8F%83%E8%80%83%E8%A7%A3%E7%AD%94_%E8%80%81%E5%B8%AB%E7%94%A8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 第 7 週 Part B 參考解答：診所掛號系統（老師用，請勿發給學生）

結構與 Part A 完全相同，只換了主題。課前請用自己的金鑰「重新啟動並執行全部」跑一次，確認 3 小時內做得完。

和 Part A 不同、值得在課堂上點出的兩個地方：
1. **關鍵字的檢查順序**：「取消掛號」裡面也有「掛號」，所以要先找「取消」（第 4 格）。
2. **安全網**：使用者自己沒說出科別名稱，就不接受 AI 推測的科別，避免系統依症狀幫人選科（第 4 格）。

In [1]:
# ============================================================
# 第 1 格：安裝套件、讀取金鑰、設定模型
# ============================================================
!pip install -q groq gradio

import sqlite3
import pandas as pd
from IPython.display import display
from google.colab import userdata
from groq import Groq
import gradio as gr

# 🔴 金鑰只能從左側「🔑 密鑰（Secrets）」讀取，絕對不可以寫成 api_key="gsk_..."
#    寫死的金鑰一旦存到 GitHub，就等於公開給全世界使用。
#    也不要 print 出金鑰，因為輸出結果也會一起存到 GitHub。
try:
    client = Groq(api_key=userdata.get('GROQ_API_KEY'))
except Exception as e:
    print("❌ 讀不到金鑰。請檢查：")
    print("   1. 左側 🔑 是否新增了名稱為 GROQ_API_KEY 的密鑰（大小寫要完全一樣）")
    print("   2. 該密鑰的「筆記本存取權」開關是否已打開")
    raise

# 模型名稱集中放在這裡；日後模型更換或下架，只要改這一行
MODEL = "openai/gpt-oss-120b"
print("✅ Groq 金鑰讀取成功，使用模型：", MODEL)

✅ Groq 金鑰讀取成功，使用模型： openai/gpt-oss-120b


In [2]:
# ============================================================
# 第 2 格：建立資料庫與初始資料
# ============================================================
DB = "clinic.db"
DEPTS = ["內科", "外科", "牙科"]
REGISTERED = "已掛號"
CANCELLED = "已取消"
NO_MEDICAL_ADVICE = "本系統不提供醫療判斷。請直接告訴我要掛哪一科（內科、外科、牙科），或洽詢櫃台人員。"

conn = sqlite3.connect(DB)
cursor = conn.cursor()
cursor.execute("DROP TABLE IF EXISTS departments")
cursor.execute("DROP TABLE IF EXISTS registrations")

cursor.execute("""
    CREATE TABLE departments (
        id    INTEGER PRIMARY KEY AUTOINCREMENT,
        name  TEXT UNIQUE NOT NULL,
        quota INTEGER NOT NULL            -- 每日名額
    )
""")
cursor.execute("""
    CREATE TABLE registrations (
        id         INTEGER PRIMARY KEY AUTOINCREMENT,
        patient    TEXT NOT NULL,
        dept       TEXT NOT NULL,
        number     INTEGER NOT NULL,      -- 看診號碼（取消後不重排）
        status     TEXT NOT NULL,         -- 已掛號 / 已取消
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
""")
# 名額故意設很少，方便測試「名額已滿」
cursor.executemany("INSERT INTO departments (name, quota) VALUES (?, ?)",
                   [("內科", 3), ("外科", 2), ("牙科", 1)])
conn.commit()
conn.close()
print("✅ 資料庫建立完成")

✅ 資料庫建立完成


In [3]:
# ============================================================
# 第 3 格：顯示資料表
# ============================================================
def show_tables():
    conn = sqlite3.connect(DB)
    print("🏥 departments（科別與名額）")
    display(pd.read_sql_query("SELECT * FROM departments", conn))
    print("📝 registrations（掛號紀錄）")
    display(pd.read_sql_query("SELECT * FROM registrations", conn))
    conn.close()

show_tables()

🏥 departments（科別與名額）


,id,name,quota
0,1,內科,3
1,2,外科,2
2,3,牙科,1


📝 registrations（掛號紀錄）


,id,patient,dept,number,status,created_at


In [4]:
# ============================================================
# 第 4 格：AI 解讀 → (動作, 科別, 原始回覆)
# ============================================================
# 🔴 順序很重要：「取消掛號」裡面也有「掛號」兩個字，所以要先找「取消」，再找「查詢」，最後才找「掛號」
ACTIONS = ["取消", "查詢", "掛號"]
dept_list = "、".join(DEPTS)

def ai_parse(user_say):
    prompt = f"""你是診所櫃台的掛號助理，只負責判斷使用者想做什麼，不提供任何醫療建議。
診所只有這些科別：{dept_list}。
請判斷使用者想「掛號」、「取消」掛號，還是「查詢」剩餘名額，以及是哪一科。

只能回覆一行，格式為：動作,科別
- 動作只能是：掛號、取消、查詢、未知
- 科別只能是：{dept_list}、未知
- 如果使用者只描述症狀、沒有說出科別名稱，科別一律回覆「未知」，不可以推測
- 不要回覆任何其他文字

範例：
使用者：我要掛內科 → 掛號,內科
使用者：幫我取消外科的掛號 → 取消,外科
使用者：牙科還有名額嗎 → 查詢,牙科
使用者：我肚子痛要掛哪科 → 掛號,未知

使用者：{user_say}"""

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        raw = response.choices[0].message.content or ""
    except Exception as e:
        print("⚠️ 呼叫 AI 失敗：", e)
        return "未知", "未知", ""

    action = "未知"
    for a in ACTIONS:
        if a in raw:
            action = a
            break

    dept = "未知"
    for d in DEPTS:
        if d in raw:
            dept = d
            break

    # ⭐ 安全網：使用者自己沒說出科別名稱，就不接受 AI 推測的科別
    if dept != "未知" and dept not in user_say:
        dept = "未知"

    return action, dept, raw

In [5]:
# ============================================================
# 第 5 格：測試清單 (使用者說的話, 預期動作, 預期科別)；"*" 表示不在意
# ============================================================
test_list = [
    ("我要掛內科",                 "掛號", "內科"),   # 正常
    ("幫我取消內科的掛號",         "取消", "內科"),   # 正常：句子裡同時有「取消」和「掛號」
    ("牙科還有名額嗎",             "查詢", "牙科"),   # 正常
    ("外科今天還能掛嗎？我想掛",   "*",    "外科"),   # 邊界：掛號還是查詢都合理
    ("我想掛眼科",                 "*",    "未知"),   # 錯誤：沒有這個科別
    ("我肚子痛，該掛哪一科？",     "*",    "未知"),   # 安全：不可依症狀推測科別
    ("今天天氣好嗎",               "未知", "未知"),   # 錯誤：無關的話
    ("我要掛外科，順便取消牙科",   "*",    "*"),      # 邊界：一句話兩件事（觀察用）
]

passed = 0
for say, exp_action, exp_dept in test_list:
    action, dept, raw = ai_parse(say)
    ok = (exp_action in ("*", action)) and (exp_dept in ("*", dept))
    passed += ok
    print(f"{'✅' if ok else '❌'} {say}")
    print(f"     AI 原始回覆：{raw!r}")
    print(f"     系統解讀：{action} / {dept}（預期：{exp_action} / {exp_dept}）\n")

print(f"通過 {passed} / {len(test_list)} 句")

✅ 我要掛內科
     AI 原始回覆：'掛號,內科'
     系統解讀：掛號 / 內科（預期：掛號 / 內科）

✅ 幫我取消內科的掛號
     AI 原始回覆：'取消,內科'
     系統解讀：取消 / 內科（預期：取消 / 內科）

✅ 牙科還有名額嗎
     AI 原始回覆：'查詢,牙科'
     系統解讀：查詢 / 牙科（預期：查詢 / 牙科）

✅ 外科今天還能掛嗎？我想掛
     AI 原始回覆：'掛號,外科'
     系統解讀：掛號 / 外科（預期：* / 外科）

✅ 我想掛眼科
     AI 原始回覆：'掛號,未知'
     系統解讀：掛號 / 未知（預期：* / 未知）

✅ 我肚子痛，該掛哪一科？
     AI 原始回覆：'掛號,未知'
     系統解讀：掛號 / 未知（預期：* / 未知）

✅ 今天天氣好嗎
     AI 原始回覆：'未知,未知'
     系統解讀：未知 / 未知（預期：未知 / 未知）

✅ 我要掛外科，順便取消牙科
     AI 原始回覆：'掛號,外科'
     系統解讀：掛號 / 外科（預期：* / *）

通過 8 / 8 句


In [6]:
# ============================================================
# 第 6 格：功能函式（掛號、取消、查詢）
# ============================================================
def register(dept, patient):
    conn = sqlite3.connect(DB)
    try:
        cursor = conn.cursor()
        cursor.execute("SELECT quota FROM departments WHERE name = ?", (dept,))
        row = cursor.fetchone()
        if row is None:
            return f"本診所沒有「{dept}」。"
        quota = row[0]

        # 同一人同一科不能重複掛號
        cursor.execute("SELECT COUNT(*) FROM registrations WHERE dept = ? AND patient = ? AND status = ?",
                       (dept, patient, REGISTERED))
        if cursor.fetchone()[0] > 0:
            return f"{patient} 今天已經掛過{dept}了，不能重複掛號。"

        # 名額檢查（只算「已掛號」的人）
        cursor.execute("SELECT COUNT(*) FROM registrations WHERE dept = ? AND status = ?",
                       (dept, REGISTERED))
        if cursor.fetchone()[0] >= quota:
            return f"不好意思，{dept}今日名額已滿。"

        # 號碼 = 這一科所有紀錄數 + 1（取消的號碼不重排）
        cursor.execute("SELECT COUNT(*) FROM registrations WHERE dept = ?", (dept,))
        number = cursor.fetchone()[0] + 1

        cursor.execute("INSERT INTO registrations (patient, dept, number, status) VALUES (?, ?, ?, ?)",
                       (patient, dept, number, REGISTERED))
        conn.commit()
        return f"✅ {patient} 掛號成功：{dept} 第 {number} 號。"
    finally:
        conn.close()


def cancel(dept, patient):
    conn = sqlite3.connect(DB)
    try:
        cursor = conn.cursor()
        cursor.execute("SELECT id, number FROM registrations WHERE dept = ? AND patient = ? AND status = ?",
                       (dept, patient, REGISTERED))
        row = cursor.fetchone()
        if row is None:
            return f"查不到 {patient} 的{dept}掛號紀錄。"
        cursor.execute("UPDATE registrations SET status = ? WHERE id = ?", (CANCELLED, row[0]))
        conn.commit()
        return f"已取消 {patient} 的{dept}第 {row[1]} 號。"
    finally:
        conn.close()


def query_dept(dept):
    conn = sqlite3.connect(DB)
    try:
        cursor = conn.cursor()
        cursor.execute("SELECT quota FROM departments WHERE name = ?", (dept,))
        row = cursor.fetchone()
        if row is None:
            return f"本診所沒有「{dept}」。"
        quota = row[0]
        cursor.execute("SELECT COUNT(*) FROM registrations WHERE dept = ? AND status = ?",
                       (dept, REGISTERED))
        used = cursor.fetchone()[0]
        return f"{dept}：今日名額 {quota}，已掛號 {used} 人，剩餘 {quota - used} 個名額。"
    finally:
        conn.close()

In [7]:
# ============================================================
# 第 7 格：主流程與完整流程測試
# ============================================================
def clinic_system(user_say, patient):
    if not user_say or not user_say.strip():
        return "請輸入你的需求（例如：我要掛內科）。"

    action, dept, raw = ai_parse(user_say)
    print(f"📌 [AI 解讀] 動作={action}，科別={dept}（原始回覆：{raw!r}）")

    if dept == "未知":
        return NO_MEDICAL_ADVICE
    if action == "查詢":
        return query_dept(dept)
    if not patient or not patient.strip():
        return "掛號或取消需要姓名，請先輸入你的姓名。"
    if action == "掛號":
        return register(dept, patient.strip())
    if action == "取消":
        return cancel(dept, patient.strip())
    return "抱歉，我不確定你要掛號、取消還是查詢，請再說清楚一點。"


cases = [
    ("我要掛牙科",         "王小明"),   # 預期：牙科第 1 號
    ("我要掛牙科",         "陳小華"),   # 預期：名額已滿（牙科只有 1 個名額）
    ("我要掛牙科",         "王小明"),   # 預期：不能重複掛號
    ("取消我的牙科掛號",   "王小明"),   # 預期：取消第 1 號
    ("我要掛牙科",         "陳小華"),   # 預期：牙科第 2 號（號碼不重排）
    ("牙科還有名額嗎",     ""),         # 預期：剩餘 0（查詢不需要姓名）
    ("我頭很痛",           "陳小華"),   # 預期：不提供醫療判斷
    ("我要掛內科",         ""),         # 預期：請先輸入姓名
]
for say, name in cases:
    print("-" * 50)
    print(f"🙋 {name or '（未填姓名）'}：{say}")
    print(clinic_system(say, name))

print("-" * 50)
show_tables()

--------------------------------------------------
🙋 王小明：我要掛牙科
📌 [AI 解讀] 動作=掛號，科別=牙科（原始回覆：'掛號,牙科'）
✅ 王小明 掛號成功：牙科 第 1 號。
--------------------------------------------------
🙋 陳小華：我要掛牙科
📌 [AI 解讀] 動作=掛號，科別=牙科（原始回覆：'掛號,牙科'）
不好意思，牙科今日名額已滿。
--------------------------------------------------
🙋 王小明：我要掛牙科
📌 [AI 解讀] 動作=掛號，科別=牙科（原始回覆：'掛號,牙科'）
王小明 今天已經掛過牙科了，不能重複掛號。
--------------------------------------------------
🙋 王小明：取消我的牙科掛號
📌 [AI 解讀] 動作=取消，科別=牙科（原始回覆：'取消,牙科'）
已取消 王小明 的牙科第 1 號。
--------------------------------------------------
🙋 陳小華：我要掛牙科
📌 [AI 解讀] 動作=掛號，科別=牙科（原始回覆：'掛號,牙科'）
✅ 陳小華 掛號成功：牙科 第 2 號。
--------------------------------------------------
🙋 （未填姓名）：牙科還有名額嗎
📌 [AI 解讀] 動作=查詢，科別=牙科（原始回覆：'查詢,牙科'）
牙科：今日名額 1，已掛號 1 人，剩餘 0 個名額。
--------------------------------------------------
🙋 陳小華：我頭很痛
📌 [AI 解讀] 動作=未知，科別=未知（原始回覆：'未知,未知'）
本系統不提供醫療判斷。請直接告訴我要掛哪一科（內科、外科、牙科），或洽詢櫃台人員。
--------------------------------------------------
🙋 （未填姓名）：我要掛內科
📌 [AI 解讀] 動作=掛號，科別=內科（原始回覆：'掛號,內科'）
掛號或取消需要姓名，請先輸入你的姓名。
-----------

,id,name,quota
0,1,內科,3
1,2,外科,2
2,3,牙科,1


📝 registrations（掛號紀錄）


,id,patient,dept,number,status,created_at
0,1,王小明,牙科,1,已取消,2026-10-27 07:22:40
1,2,陳小華,牙科,2,已掛號,2026-10-27 07:22:42


In [8]:
# ============================================================
# 第 8 格：Gradio 網頁
# ============================================================
demo = gr.Interface(
    fn=clinic_system,
    inputs=[
        gr.Textbox(label="💬 你的需求（例如：我要掛內科／牙科還有名額嗎）"),
        gr.Textbox(label="🙋 你的姓名（查詢可不填）"),
    ],
    outputs=gr.Textbox(label="🏥 系統回應"),
    title="🏥 診所掛號系統（簡化版）",
    description="可掛號、取消、查詢剩餘名額。本系統不提供任何醫療建議。",
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d9ddea3acd8461b2ff.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
